# K-Nearest Neighbor Classifier

- K-Nearest Neighbor Classifier (KNN)
  - Is very simple to implement
  - Can model non-linear decision boundaries
  - Can be really hard to beat even for big neural networks

## Linear Classification

<table><tr>
<td style="width:50%; text-align:center; vertical-align:middle;">

![hyperplane](figures/hyperplane.pdf)

</td>
<td style="width:50%; vertical-align:middle;">

- What is a good $\mathbf{w}$?
- Some proposals:
  - Logistic Regression
  - Perceptrons
  - Support Vector Machines
  - Ridge Regression
- Linear methods have different ways of defining what a good $\mathbf{w}$ is

</td>
</tr></table>

## Linear Classification

<table><tr>
<td style="width:50%; text-align:center; vertical-align:middle;">

![hyperplane](figures/hyperplane.pdf)

</td>
<td style="width:50%; vertical-align:middle;">

Nearest Centroid Classification is a simple linear classifier

$$\mathbf{w} = \boldsymbol{\mu}_o - \boldsymbol{\mu}_\Delta$$

$$\beta = -\tfrac{1}{2}(\boldsymbol{\mu}_o^{\top}\boldsymbol{\mu}_o - \boldsymbol{\mu}_\Delta^{\top}\boldsymbol{\mu}_\Delta)$$

</td>
</tr></table>

## Problems with Linear Classification

<div style="text-align:center;">

Linear Classifiers fail on non-linear problems:

<table><tr>
<td style="text-align:center;">

![perceptron_problem_1](figures/perceptron_problem_1.pdf)

</td>
<td style="text-align:center;">

![perceptron_problem_2](figures/perceptron_problem_2.pdf)

</td>
</tr></table>

Try to separate these two-class data sets with **a single line**

</div>

## Problems with Linear Classification

<div style="text-align:center;">

If we can model the non-linearity,  
we can create new features $\tilde{\mathbf{x}}$ which are linearly separable

<br>

<table><tr>
<td style="text-align:center;">

![non-linear-features-1](figures/non-linear-features-1.png)

</td>
<td style="text-align:center;">

![non-linear-features-2](figures/non-linear-features-2.png)

</td>
</tr></table>

But what if we do not know the non-linearity?

</div>

## K-Nearest Neighbor Classifier

- Nearest Centroid Classifiers require estimation of centroids
- K-Nearest Neighbor is simpler
- Idea:
  1. Find the $k$ closest neighbors for a new data point $\mathbf{x}$
  2. Look up labels of $k$ closest neighbors
  3. Assign majority vote label for $\mathbf{x}$

## K-Nearest Neighbor Classifier

<div style="text-align:center;">

We do not need to train a model for KNN --  
**the data is the model**  
But (as with NCC) we need a distance function  
Usually the euclidean distance is chosen:

$$d(\mathbf{x}_i, \mathbf{x}_j) = \|\mathbf{x}_i - \mathbf{x}_j\|_2 \qquad j \in [1, \dots, N]$$

where $N$ is the number of data points

</div>

## K-Nearest Neighbor Classifier: Pseudocode

**Algorithm: K-Nearest Neighbour Prediction**

**Input:**  
Test data point $\mathbf{x}_i \in \mathbb{R}^{D}$,  
training data $\mathbf{X} = [\mathbf{x}_1, \dots, \mathbf{x}_N] \in \mathbb{R}^{D \times N}$,  
labels $\mathbf{y} = [y_1, \ldots, y_N]^{\top} \in \mathbb{R}^N$,  
number of neighbours $K$

**Output:** Predicted label $\mathbf{y}_i$

---

* **for** $\mathbf{x}_j$ in $\mathbf{X}_{\text{train}}$:  
    * Compute distance between test point $\mathbf{x}_i$ and training point $\mathbf{x}_j$
* Sort Training data points by distance to test point $\mathbf{x}_i$
* Pick closest $k$ neighbors
* Compute most frequent label amongst neighbors
* Return majority label (ties are broken at random)  


## KNN Implementation

In [1]:
class KNN:
    def __init__(self, X_train, y_train, k=5):
        self.X_train = X_train
        self.y_train = y_train
        self.classes = np.unique(y_train)
        self.k = k
    
    def predict(self, X_test):
        y_pred = []
        for x in X_test: 
            # find indices of nearest neighbors
            nn_idx = np.linalg.norm(self.X_train - x, axis=1).argsort()[:self.k]
            # count labels of nearest neighbors
            vals, counts = np.unique(self.y_train[nn_idx], return_counts=True)
            # predict and append majority label
            y_pred.append(vals[counts.argmax()])
        return np.array(y_pred)


## K-Nearest Neighbor Classifier

<div style="text-align:center;">

Toy data problem: Linear classification

<table><tr>
<td style="text-align:center;">

![knn-k1](figures/knn-k1.pdf)

$k=1$

</td>
<td style="text-align:center;">

![knn-k15](figures/knn-k15.pdf)

$k=15$

</td>
</tr></table>

</div>

In [2]:
import matplotlib.pylab as plt
import numpy as np
from numpy.random import multivariate_normal as mvn
%matplotlib inline
# %config InlineBackend.figure_format = 'svg'
import matplotlib
# matplotlib.use('Agg')
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (5.0, 5.0)
from sklearn.datasets import make_moons, make_blobs
from sklearn.model_selection import train_test_split

np.random.seed(0)

def accuracy(ypred,ytrue):
    return (ypred==ytrue).mean()

def plot_data_and_model_predictions(X_train, y_train, X_test, y_test, model=None):
    if model:
        # Plot the decision boundary.
        h = .1 # stepsize in mesh
        offset = .1
        offset = .1
        x_min, x_max = np.vstack([X_train,X_test])[:, 0].min() - offset, np.vstack([X_train,X_test])[:, 0].max() + offset
        y_min, y_max = np.vstack([X_train,X_test])[:, 1].min() - offset, np.vstack([X_train,X_test])[:, 1].max() + offset
        xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
        Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
        Z = Z.reshape(xx.shape)
        cs = plt.contourf(xx, yy, Z, cmap=plt.cm.cividis, alpha=.6)

    plt.plot(X_train[y_train==0,0],X_train[y_train==0,1],'b.',
         X_test[y_test==0,0],X_test[y_test==0,1],'bo',
         X_train[y_train==1,0],X_train[y_train==1,1],'r.',
         X_test[y_test==1,0],X_test[y_test==1,1],'ro')
    
    plt.xlabel('$X_1$')
    plt.ylabel('$X_2$')
    plt.legend(['Class 0 Train','Class 0 Test' ,'Class 1 Train','Class 1 Test'])
    if model:
        cbar = plt.colorbar()
        cbar.set_label('p(y=1|x)')
    plt.axis('tight')
    if model:
        acc_train = accuracy(model.predict(X_train), y_train)
        acc_test = accuracy(model.predict(X_test), y_test)
        plt.title(f'Acc (train/test): {acc_train:0.2}/{acc_test}')
    plt.show()

## KNN on simple linear toy data


In [3]:
n_samples = 100
test_size = .2
noise = .25

X_linear, y_linear = make_blobs(n_samples=n_samples, 
                                n_features=2, 
                                centers=np.array([[1,1],[-1,-1]]), 
                                cluster_std=np.array([1.,1.])*noise)

X_linear_train, X_linear_test, y_linear_train, y_linear_test = \
    train_test_split(X_linear, y_linear, test_size=test_size)


In [ ]:
knn = KNN(X_linear_train, y_linear_train, k=1)
plot_data_and_model_predictions(X_linear_train, 
                                y_linear_train, 
                                X_linear_test, 
                                y_linear_test,knn)
y_linear_test_pred = knn.predict(X_linear_test)

## KNN on simple non-linear toy data


In [ ]:
n_samples = 100
test_size = .2
noise = .1

X_nonlinear, y_nonlinear = make_moons(n_samples=n_samples, noise=noise)
X_nonlinear_train, X_nonlinear_test, y_nonlinear_train, y_nonlinear_test = \
    train_test_split(X_nonlinear, y_nonlinear, test_size=test_size)


In [ ]:
knn = KNN(X_nonlinear_train, y_nonlinear_train, k=1)
plot_data_and_model_predictions(X_nonlinear_train, 
                                y_nonlinear_train, 
                                X_nonlinear_test, 
                                y_nonlinear_test,knn)
y_nonlinear_test_pred = knn.predict(X_nonlinear_test)

## KNN on noisy linear toy data

> KNN is overfitting

In [ ]:
n_samples = 100
test_size = 2
noise = 2
k = 1

X_linear, y_linear = make_blobs(n_samples=n_samples, 
                                n_features=2, 
                                centers=np.array([[1,1],[-1,-1]]), 
                                cluster_std=np.array([1.,1.])*noise)

X_linear_train, X_linear_test, y_linear_train, y_linear_test = \
    train_test_split(X_linear, y_linear, test_size=test_size)


In [ ]:
knn = KNN(X_linear_train, y_linear_train, k=k)
plot_data_and_model_predictions(X_linear_train, 
                                y_linear_train, 
                                X_linear_test, 
                                y_linear_test,knn)
y_linear_test_pred = knn.predict(X_linear_test)

## Controling complexity

> Hyperparameter $k$ controls complexity of decision boundary


In [ ]:
n_samples = 100
test_size = .2
noise = 2
k = 30

X_linear, y_linear = make_blobs(n_samples=n_samples, 
                                n_features=2, 
                                centers=np.array([[1,1],[-1,-1]]), 
                                cluster_std=np.array([1.,1.])*noise)

X_linear_train, X_linear_test, y_linear_train, y_linear_test = \
    train_test_split(X_linear, y_linear, test_size=test_size)


In [ ]:
knn = KNN(X_linear_train, y_linear_train, k=k)
plot_data_and_model_predictions(X_linear_train, 
                                y_linear_train, 
                                X_linear_test, 
                                y_linear_test,knn)

## KNN and the XOR Problem

* Remember Marvin Minsky's XOR Problem from the NCC?
* KNN happily solves that, too
* But how to set the hyperparameter $k$?

<div style="text-align:center;">

<table><tr>
<td style="text-align:center;">

![knn-k1_xor](figures/knn-k1_xor.pdf)

$k=1$

</td>
<td style="text-align:center;">

![knn-k15_xor](figures/knn-k15_xor.pdf)

$k=15$

</td>
</tr></table>

</div>

## Problems with KNN

- Hyperparameter $k$ needs to be set appropriately:
  - $k$ small: complex decision boundaries
  - $k$ large: smooth/simple decision boundaries
- Consider $N$ data points $\mathbf{x} \in \mathbb{R}^D$
- Finding $K$ neighbors requires $\mathcal{O}(NND)$ operations
- For large data sets this is too costly
- Speedups can be gained by:
  - Trees for distance computations
  - Locality Sensitive Hashing for finding neighbors

# Hyperparameter Optimization

We use grid search to find the best k parameter.

## Overfitting

- Hyperparameter $k$ controls complexity of KNN predictions
- Often data is complex because it's noisy $\rightarrow$ $k$ too small leads to **overfitting**
- Overfitting can be thought of memorizing the training data (including its noise)
- But we don't want to predict noise
- We want high accuracy on test data (that our model hasn't seen yet)
- In other words: we want the model to **generalize**
- Decreasing $k$ will *always improve* accuracy on **training data**


>How can we learn $k$ such that KNN has  
the right complexity for generalizing to new data?


## Model Selection

- $k$ can be learned by **Hyperparameter Optimization**
- Several options:
  - Grid Search
  - Random Search
  - Bayesian Global Optimization

## Grid Search with Cross-Validation

- Define a **grid** of hyperparameter candidates $k \in \{1, 2, 3, \ldots\}$

- Split data set in $F$ different **training** and **test** data folds:

$$\left[\; \underbrace{x_1,\; x_2,\; x_3,\; x_4}_{\text{Training Data}},\; \underbrace{x_5,\; x_6}_{\text{Test Data}} \;\right]$$

- For each $k$ in grid:
  - **Train** your model on the training data $\mathcal{F}^{\,\text{train}}$
  - **Test** your model on the test data $\mathcal{F}^{\,\text{test}}$

- Then choose the $k$ that worked best on the held-out test data.

In [ ]:
n_samples = 100
test_size = .5
noise = 2.5

X_nonlinear, y_nonlinear = make_moons(n_samples=n_samples, noise=noise)
X_nonlinear_train, X_nonlinear_test, y_nonlinear_train, y_nonlinear_test = \
    train_test_split(X_nonlinear, y_nonlinear, test_size=test_size)


In [ ]:
acc_test = []
acc_train = []

for k in range(1,20):
    knn = KNN(X_nonlinear_train, y_nonlinear_train, k=k)
    y_train_pred = knn.predict(X_nonlinear_train)
    y_test_pred = knn.predict(X_nonlinear_test)
    acc_test.append((k,accuracy(y_nonlinear_test, y_test_pred)))
    acc_train.append((k,accuracy(y_nonlinear_train, y_train_pred)))

In [ ]:
k, val_acc = zip(*acc_test)
_, train_acc = zip(*acc_train)
plt.plot(k, val_acc,'o-', k, train_acc,'x-')
plt.xlabel('k')
plt.ylabel('Accuracy on Test Data')
plt.title('Linear Classification')
plt.legend(['validation data', 'train data'])
plt.savefig('figures/knn-val-vs-train.pdf')

![cross-validation](figures/knn-val-vs-train.pdf)

## Summary KNN Classifier

* Simple nonlinear classifier
* Requires no training
* But appropriate features
* And a distance function
* Can yield state-of-the-art prediction performance
* Needs to evaluate pairwise distances of all data points
* $\rightarrow$ **Slow**
* $\rightarrow$ **Won't scale to large data sets**